# Replication: Vector Arithmetic in Concept and Token Subspaces

## Overview
This notebook replicates the experiment from the paper "Vector Arithmetic in Concept and Token Subspaces" (NeurIPS 2025 Mechanistic Interpretability Workshop).

### Key Hypothesis
1. Poor parallelogram arithmetic on raw Llama-2-7b hidden states is due to interference from irrelevant information
2. Word2vec arithmetic is only effective in a semantic subspace, not on full hidden states
3. Concept and token induction heads operate in different subspaces (semantic vs. surface-level)

### Methodology
1. Build lenses by summing OV matrices from top-k concept/token induction heads
2. Extract word embeddings through Llama-2-7b at layer ℓ, transformed using lens matrices
3. Test parallelogram arithmetic: a - b + b' and check if a' is nearest neighbor
4. Compare: raw, concept lens, token lens, and all heads

In [1]:
# Setup and imports
import os
import json
import torch
import numpy as np
from typing import Dict, List, Tuple, Optional

os.chdir('/home/smallyan/eval_agent')

# Set device and seeds
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
np.random.seed(42)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

# Paths
repo_path = '/net/scratch2/smallyan/arithmetic_eval'
cache_path = os.path.join(repo_path, 'cache')
data_path = os.path.join(repo_path, 'data')

CUDA available: True
Device: NVIDIA A100 80GB PCIe


In [2]:
# Load nnsight and model
from nnsight import LanguageModel

print("Loading Llama-2-7b model...")
model = LanguageModel('meta-llama/Llama-2-7b-hf', device_map='cuda', dispatch=True)
print(f"Model loaded! Hidden size: {model.config.hidden_size}, Layers: {model.config.num_hidden_layers}")

Loading Llama-2-7b model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded! Hidden size: 4096, Layers: 32


In [3]:
# Load head ordering scores
llama2_cache = os.path.join(cache_path, 'causal_scores', 'Llama-2-7b-hf')

with open(os.path.join(llama2_cache, 'concept_copying_len30_n1024.json'), 'r') as f:
    concept_scores = json.load(f)
with open(os.path.join(llama2_cache, 'token_copying_len30_n1024.json'), 'r') as f:
    token_scores = json.load(f)

# Sort by score to get top-k heads
concept_sorted = sorted([(d['layer'], d['head_idx'], d['score']) for d in concept_scores], 
                        key=lambda t: t[2], reverse=True)
token_sorted = sorted([(d['layer'], d['head_idx'], d['score']) for d in token_scores], 
                      key=lambda t: t[2], reverse=True)

print(f"Top 5 concept heads: {concept_sorted[:5]}")
print(f"Top 5 token heads: {token_sorted[:5]}")

Top 5 concept heads: [(14, 1, 0.0010720472782850266), (14, 9, 0.0003007464110851288), (11, 22, 0.0002677999436855316), (13, 23, 0.00024266168475151062), (9, 25, 0.0001970883458852768)]
Top 5 token heads: [(16, 19, 0.0021728137508034706), (11, 15, 0.0012158188037574291), (11, 2, 0.000996168702840805), (12, 26, 0.0009576366282999516), (8, 26, 0.0007893270812928677)]


## 2. Core Functions Implementation

In [4]:
def get_ov_sum(model, head_ordering: str = 'concept', k: int = 80, rank: int = 4096) -> Optional[torch.Tensor]:
    """Construct the lens matrix by summing OV matrices from top-k heads."""
    head_dim = model.config.hidden_size // model.config.num_attention_heads
    
    if head_ordering == 'raw':
        return None
    elif head_ordering == 'all':
        to_sum = [(l, h) for l in range(model.config.num_hidden_layers) 
                  for h in range(model.config.num_attention_heads)]
    else:
        sorted_heads = concept_sorted if head_ordering == 'concept' else token_sorted
        to_sum = [(l, h) for l, h, _ in sorted_heads][:k]
    
    with torch.no_grad():
        ov_sum = torch.zeros((4096, 4096), device='cuda')
        for l, h in to_sum:
            V = model.model.layers[l].self_attn.v_proj.weight[h * head_dim : (h+1) * head_dim]
            O = model.model.layers[l].self_attn.o_proj.weight[:, h * head_dim : (h+1) * head_dim]
            ov_sum += torch.matmul(O, V)
        
        if rank < model.config.hidden_size:
            U, S, Vh = torch.linalg.svd(ov_sum)
            ov_sum = (U[:, :rank] * S[:rank]) @ Vh[:rank]
    
    return ov_sum

def get_word_rep(word: str, model, layer_idx: int, ov_sum: Optional[torch.Tensor] = None, 
                 prefix: str = '') -> torch.Tensor:
    """Get word representation at a given layer, optionally transformed by OV sum."""
    text = prefix + word.strip()
    with torch.no_grad():
        with model.trace(text):
            state = model.model.layers[layer_idx].output[0].squeeze()[-1].detach().save()
    if ov_sum is not None:
        return torch.matmul(ov_sum, state)
    return state

def get_all_neighbors(task_lines: List[str], model, layer_idx: int, 
                      ov_sum: Optional[torch.Tensor] = None, prefix: str = '', sep: str = ' '):
    """Get representations for all unique words in a task."""
    words = set()
    for line in task_lines:
        words.update(line.split(sep))
    return {word: get_word_rep(word, model, layer_idx, ov_sum, prefix) for word in words}

print("Core functions defined")

Core functions defined


In [5]:
def compute_nn_accuracy(task_lines: List[str], neighbors: Dict[str, torch.Tensor], sep: str = ' '):
    """Compute parallelogram arithmetic accuracy using nearest neighbor matching."""
    correct = 0
    total = 0
    
    for line in task_lines:
        parts = line.split(sep)
        if len(parts) != 4:
            continue
        a, b, a_prime, b_prime = parts
        
        # Compute: a - b + b' should equal a'
        result = neighbors[a] - neighbors[b] + neighbors[b_prime]
        
        # Find nearest neighbor
        best_word = max(neighbors.keys(), 
                        key=lambda w: torch.cosine_similarity(result, neighbors[w], dim=0).item())
        
        if best_word == a_prime:
            correct += 1
        total += 1
    
    return correct / total if total > 0 else 0, total

def load_task(task_name: str, dataset: str = 'word2vec') -> List[str]:
    """Load task data from file."""
    filepath = os.path.join(data_path, dataset, f'{task_name}.txt')
    with open(filepath, 'r') as f:
        content = f.read()
    return [l for l in content.split('\n')[1:] if l.strip()]

print("Evaluation functions defined")

Evaluation functions defined


## 3. Run Experiments

We will replicate the key experiments:
1. Capital Cities task (semantic - concept lens should excel)
2. Present Participle task (grammatical - token lens should excel)
3. Multiple layers comparison

In [6]:
# Run experiment for a single task/layer/ordering
def run_experiment(task_name: str, layer: int, head_ordering: str, k: int = 80, 
                   dataset: str = 'word2vec', prefix: str = ''):
    """Run parallelogram experiment for a specific configuration."""
    task_lines = load_task(task_name, dataset)
    sep = ' ' if dataset == 'word2vec' else '\t'
    
    ov_sum = get_ov_sum(model, head_ordering, k)
    neighbors = get_all_neighbors(task_lines, model, layer, ov_sum, prefix, sep)
    acc, n = compute_nn_accuracy(task_lines, neighbors, sep)
    
    return {'nn_acc': acc, 'n': n, 'layer': layer, 'ordering': head_ordering}

# Test with capital-common-countries at layer 20
print("Testing experiment function...")
result = run_experiment('capital-common-countries', layer=20, head_ordering='concept')
print(f"Concept lens, layer 20: accuracy = {result['nn_acc']:.3f} (n={result['n']})")

Testing experiment function...


You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Concept lens, layer 20: accuracy = 0.895 (n=506)


In [7]:
# Run full experiment across layers and orderings for capital-common-countries
layers = [0, 4, 8, 12, 16, 20, 24, 28, 31]
orderings = ['raw', 'concept', 'token', 'all']

results_capital = {}
print("Running capital-common-countries experiments...")

for ordering in orderings:
    results_capital[ordering] = {}
    print(f"\n{ordering}:", end=" ")
    for layer in layers:
        result = run_experiment('capital-common-countries', layer=layer, head_ordering=ordering)
        results_capital[ordering][layer] = result['nn_acc']
        print(f"L{layer}={result['nn_acc']:.3f}", end=" ")

print("\n\nResults summary:")

Running capital-common-countries experiments...

raw: 

L0=0.000 

L4=0.024 

L8=0.051 

L12=0.091 

L16=0.172 

L20=0.158 